# Lecture 10 — Vanishing Gradients & LSTMs, made visible

In the lecture you saw *why* a plain RNN forgets: backprop through time multiplies one factor per step, so the gradient reaching an early step scales like `rᵀ` — it **vanishes** (`r<1`) or **explodes** (`r>1`). This notebook makes that concrete and then shows the fix:

1. the **`rᵀ` mechanism** in three lines of Python,
2. **measure the gradient actually vanish** through a real `tanh` RNN (this *is* the problem, on a log axis),
3. the **LSTM cell-state highway** — the *same* measurement, gradient now **survives** (the "constant error carousel"),
4. **inside the cell** — a from-scratch gate that stores / holds / overwrites,
5. **`nn.LSTM` / `nn.GRU`** in PyTorch, and the **4× parameter** price of the highway,
6. **exploding gradients & gradient clipping**.

It's deliberately small and **deterministic** — every number and plot is reproducible, and it runs fine on CPU (no GPU needed for this one).

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)
print('(This notebook is tiny and deterministic — CPU is perfectly fine.)')

## 1. The mechanism: gradient ∝ rᵀ

Strip backprop-through-time down to a single repeated factor `r`. After `T` steps the gradient is `r**T`. Three fates:

In [ ]:
def propagate(r, steps=25):
    g = 1.0
    for _ in range(steps):
        g *= r          # one BPTT factor per timestep
    return g

for r in (0.5, 1.0, 1.5):
    print(f'r={r}:  r^25 = {propagate(r):.2e}')

`0.5²⁵ ≈ 3e-8` (vanished), `1.5²⁵ ≈ 25000` (exploded), `1.0` holds — but only on a knife-edge. Plotted on a **log axis**, the three fates are unmistakable:

In [ ]:
Ts = range(0, 61)
for r, style in [(0.5, 'C0'), (1.0, 'C2'), (1.5, 'C3')]:
    plt.plot(list(Ts), [r**t for t in Ts], style, label=f'r = {r}')
plt.yscale('log'); plt.xlabel('timesteps back, T'); plt.ylabel('gradient ∝ r^T (log)')
plt.axhline(1, color='k', lw=.5); plt.legend(); plt.title('Why long-range gradients vanish or explode')
plt.show()

That flat green line (`r≈1`) is exactly what the LSTM's cell-state path will buy us — a per-step factor near 1 instead of a `tanh`-derivative well below it. Let's first watch a real RNN fall off the blue line.

## 2. Watch the gradient vanish — through a real `tanh` RNN

No toy scalar now: a genuine `nn.RNNCell` (`hₜ = tanh(Wₓxₜ + W_h hₜ₋₁ + b)`), unrolled for `T` steps. We put a loss on the **last** hidden state and ask: how big is the gradient `∂loss/∂hₜ` at each earlier step `t`? If it shrinks toward zero as we go back, early steps get **no learning signal** — that's the vanishing gradient, measured.

In [ ]:
def rnn_grad_decay(T=60, H=64, seed=0):
    torch.manual_seed(seed)
    cell = nn.RNNCell(H, H)
    x = torch.zeros(T, 1, H)                 # zero inputs: isolate the recurrent path
    h = torch.zeros(1, H)
    hs = []
    for t in range(T):
        h = cell(x[t], h)
        h.retain_grad()                      # keep ∂loss/∂h_t for every step
        hs.append(h)
    hs[-1].pow(2).sum().backward()           # loss depends on the LAST hidden state
    return [hs[t].grad.norm().item() for t in range(T)]

g = rnn_grad_decay()
steps_back = list(range(len(g)-1, -1, -1))   # 0 = last step
print(f'grad norm at the last step: {g[-1]:.3f}')
print(f'   30 steps back: {g[-30]:.2e}')
print(f'   55 steps back: {g[-55]:.2e}')

In [ ]:
plt.plot(steps_back, g, 'C0.-')
plt.yscale('log'); plt.gca().invert_xaxis()
plt.xlabel('timesteps back from the loss'); plt.ylabel('‖∂loss/∂hₜ‖ (log)')
plt.title('A plain RNN: the gradient vanishes going back in time'); plt.show()

There it is — the gradient falls off a cliff, ~`1.7` at the last step down to ~`10⁻¹³` a few dozen steps back. An early input simply **cannot** influence a late loss: the signal to update it has decayed to nothing. This is not a bug; it's the default behaviour of a `tanh` RNN.

## 3. The LSTM highway — the gradient survives

The LSTM adds a **cell state** `cₜ` updated mostly by *addition*: `cₜ = fₜ·cₜ₋₁ + iₜ·gₜ`. When the **forget gate** `fₜ ≈ 1`, we have `∂cₜ/∂cₜ₋₁ = fₜ ≈ 1` — no shrinking. Run the *exact same measurement* down the cell state of an `nn.LSTMCell` (with the forget gate biased open, the standard trick) and compare:

In [ ]:
def lstm_cell_grad_decay(T=60, H=64, seed=0):
    torch.manual_seed(seed)
    cell = nn.LSTMCell(H, H)
    with torch.no_grad():
        cell.bias_ih[H:2*H] += 3.0           # open the forget gate (f ≈ 0.95) — 'remember by default'
    x = torch.zeros(T, 1, H)
    h, c = torch.zeros(1, H), torch.zeros(1, H)
    cs = []
    for t in range(T):
        h, c = cell(x[t], (h, c))
        c.retain_grad()                      # gradient along the cell-state highway
        cs.append(c)
    cs[-1].pow(2).sum().backward()
    return [cs[t].grad.norm().item() for t in range(T)]

gl = lstm_cell_grad_decay()
plt.plot(steps_back, g,  'C0.-', label='plain RNN (hidden state)')
plt.plot(steps_back, gl, 'C2.-', label='LSTM (cell state)')
plt.yscale('log'); plt.gca().invert_xaxis()
plt.xlabel('timesteps back from the loss'); plt.ylabel('gradient norm (log)')
plt.legend(); plt.title('Same measurement: RNN vanishes, LSTM cell state survives'); plt.show()
print(f'55 steps back —  RNN: {g[-55]:.1e}   LSTM: {gl[-55]:.1e}')

The blue line dives; the green line stays **flat** across all 60 steps. That flat line is Hochreiter's **constant error carousel**: because the cell state is carried by addition with `f≈1`, the gradient rides back through time almost untouched. *This* is why an LSTM can connect a cause now to an effect 50 steps later, and a plain RNN can't.

### Why the highway factor is ≈ 1 — watch backprop

*Why* does the green line stay flat? Compare the per-step gradient factor of the two updates.

- **Plain RNN (multiplicative):** `hₜ = tanh(W_h hₜ₋₁)`, so `∂hₜ/∂hₜ₋₁ = tanh′ · W_h`. That's a `tanh′` (**≤ 1**, and → 0 when saturated) times a **learned matrix reused every step** — nothing pins it near 1, and chaining it is `(tanh′·W_h)ᵀ` → vanish or explode.
- **LSTM cell (additive):** `cₜ = fₜ·cₜ₋₁ + iₜ·gₜ`, so along the carry `∂cₜ/∂cₜ₋₁ = fₜ` — just the **forget gate**. The `+ iₜ·gₜ` term is *added*, so it contributes **nothing** to the gradient of the carry.

That single factor `fₜ` is **stable** for three reasons: it's a sigmoid so it's **capped at 1 (can't explode)**; it's **learnable**, so the net drives `fₜ → 1` to remember; and `cₜ = cₜ₋₁ + …` when `f≈1` is a **skip/identity path through time** (the same trick that lets ResNets go deep). Let autograd report both factors:

In [ ]:
# ONE step: what gradient does backprop send to the carry?
f, i, g = 0.95, 0.6, 0.5
c_prev = torch.tensor(2.0, requires_grad=True)
c_t = f * c_prev + i * g                        # additive update
c_t.backward()
print(f'additive  cₜ=f·cₜ₋₁+i·g :  ∂cₜ/∂cₜ₋₁ = {c_prev.grad.item():.2f}   (= f; the +i·g term adds 0)')

w = 1.2
h_prev = torch.tensor(0.8, requires_grad=True)
h_t = torch.tanh(w * h_prev)                    # multiplicative tanh update
h_t.backward()
print(f'tanh      hₜ=tanh(w·hₜ₋₁):  ∂hₜ/∂hₜ₋₁ = {h_prev.grad.item():.2f}   (= w·tanh′ < 1)')

In [ ]:
# CHAINED over 50 steps — autograd multiplies the per-step factors for us
def chain_grad(step, x0=1.0, T=50):
    x = torch.tensor(x0, requires_grad=True)
    z = x
    for _ in range(T):
        z = step(z)
    z.backward()
    return x.grad.item()

print('additive, f=0.95 :  ∂c₅₀/∂c₀ =', round(chain_grad(lambda c: 0.95*c + 0.3), 4), '  (0.95^50; capped ≤ 1)')
print('additive, f=0.999:  ∂c₅₀/∂c₀ =', round(chain_grad(lambda c: 0.999*c),   4), '  (remember-gate ≈ open → barely decays)')
print('tanh RNN, w=1.2  :  ∂h₅₀/∂h₀ =', f'{chain_grad(lambda h: torch.tanh(1.2*h), 0.5):.1e}', '  (vanished)')

So the LSTM's carry gradient is a product of **forget gates** (`∏ fₜ`) instead of `tanh′·W` factors: bounded above by 1 (no explosion) and held near 1 when the net wants to remember (no vanishing). With `f=0.999` the gradient after 50 steps is still `0.95`; the `tanh` RNN has already vanished to `~10⁻⁸`. **That** is the highway.

## 4. Inside the cell — store, hold, overwrite

The gates are just `sigmoid`s in `[0,1]` that multiply a signal (**0 blocks, 1 passes**). With an *independent* forget gate `f` and input gate `i`, the update `c = f·c + i·g` can **store** a value, **hold** it untouched for many steps, then **overwrite** it — exactly what memory needs. Here it is from scratch (stdlib-style), matching the lecture:

In [ ]:
import math
def sigmoid(z): return 1/(1+math.exp(-z))

c = 0.0
# (forget logit, input logit, candidate g):  store -> hold -> hold -> overwrite
program = [(2.0, 6.0, 0.8), (6.0, -6.0, 0.0), (6.0, -6.0, 0.0), (-6.0, 6.0, -0.5)]
labels  = ['store 0.8', 'hold', 'hold', 'overwrite -0.5']
o_logit = 4.0                                # output gate: how much of the cell to expose
for (fl, il, gg), lab in zip(program, labels):
    f, i, o = sigmoid(fl), sigmoid(il), sigmoid(o_logit)
    c = f*c + i*gg                           # additive cell update, independent gates
    h = o * math.tanh(c)                     # what the rest of the net sees: h = o·tanh(c)
    print(f'{lab:16s}  f={f:.2f} i={i:.2f}  ->  c = {c:+.3f}   h = {h:+.3f}')

While the value is *held*, `f≈1` so `c` barely drifts (`0.798 → 0.794`) **and** its gradient `∂cₜ/∂cₜ₋₁ = f ≈ 1` survives — value and learning signal both preserved. Now contrast the two gradient products directly: a plain RNN multiplies a factor `r<1`; the LSTM multiplies forget gates `f≈1`.

In [ ]:
T, r = 30, 0.6
f = sigmoid(4.0)                             # forget gate ≈ 0.982
rnn = lstm = 1.0
for _ in range(T):
    rnn  *= r                                # plain recurrence: r^T
    lstm *= f                                # cell-state path: product of forget gates
print(f'after {T} steps —  RNN r^T = {rnn:.2e}   LSTM ∏f = {lstm:.2f}')

## 5. Using it for real — `nn.LSTM`, and the 4× price

You never hand-roll this. `nn.LSTM` is batched and optimised; for classification you feed the **last** timestep's hidden state into a linear head.

In [ ]:
lstm = nn.LSTM(input_size=16, hidden_size=64, num_layers=1, batch_first=True)
x = torch.randn(8, 30, 16)                   # (batch, seq_len, features)
out, (h_n, c_n) = lstm(x)
print('out:', tuple(out.shape), '  h_n:', tuple(h_n.shape), '  c_n:', tuple(c_n.shape))
print('last-step hidden state for a classifier head:', tuple(out[:, -1, :].shape))

The highway isn't free: an LSTM computes **four** gated transforms per step (forget, input, output, candidate) where a plain RNN computes **one** — so ~**4× the parameters** per layer (a GRU: **3×**). That extra 4× *is* the price of the additive gradient highway.

In [ ]:
d, H = 16, 64
for name, mod in [('RNN', nn.RNN(d, H)), ('GRU', nn.GRU(d, H)), ('LSTM', nn.LSTM(d, H))]:
    n = sum(p.numel() for p in mod.parameters())
    print(f'{name:5s} {n:6d} params   ({n / 5248:.0f}x the RNN)')   # 5248 = the RNN's count

## 6. The GRU — a leaner gate

The **GRU** keeps the gating idea with **no separate cell state** and only **two** gates (update, reset) — fewer parameters, often trains a bit faster, accuracy usually close to an LSTM's. Same API, minus `c_n`. LSTM vs GRU is an **empirical** choice — try both.

In [ ]:
gru = nn.GRU(input_size=16, hidden_size=64, batch_first=True)
out, h_n = gru(x)                            # note: no c_n
print('GRU out:', tuple(out.shape), '  h_n:', tuple(h_n.shape))

## 7. Exploding gradients — clip them

Gates cure *vanishing*; **exploding** gradients can still strike on an unlucky batch. The cheap, standard fix is **gradient clipping**: if the global gradient norm exceeds a threshold, rescale it down — **same direction, capped length**, so one spike can't blow up training.

In [ ]:
# a parameter with a huge gradient (a 'spike')
p = nn.Parameter(torch.zeros(3))
p.grad = torch.tensor([12.0, -9.0, 5.0])
before = p.grad.norm().item()
torch.nn.utils.clip_grad_norm_([p], max_norm=1.0)
after = p.grad.norm().item()
print(f'grad norm  {before:.2f}  ->  {after:.2f}   (capped at 1.0; direction unchanged: {p.grad / after})')

# in a real training loop:
#   loss.backward()
#   torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#   optimizer.step()

> **Two diseases, two cures.** *Vanishing* → change the **architecture** (LSTM/GRU, or shorten the sequence). *Exploding* → **clip the gradient**. `max_norm` of `1.0`–`5.0` is a sensible RNN default.

## Your turn

1. **Depth of the cliff.** In §2, change the recurrent weight scale (e.g. multiply `cell.weight_hh` by `0.5` vs `2.0` under `torch.no_grad()`) and re-plot. When does the RNN gradient vanish faster? When can it *explode*?
2. **Open vs shut forget gate.** In §3, drop the `+= 3.0` forget-gate bias to `0.0` (gate ≈ 0.5). Does the LSTM cell-state gradient still survive 60 steps, or start to decay? Why?
3. **Leaky memory.** Rewrite §4 in the GRU-style tied form `c = f·c + (1−f)·new` and watch a held value slowly leak when `f` isn't near 1.
4. **Count by hand.** For `input=16, hidden=128`, predict the LSTM and GRU parameter counts with `≈4·H(d+H)` and `≈3·H(d+H)`, then check against `nn.LSTM`/`nn.GRU`.

## Recap
- Backprop-through-time multiplies one factor per step → gradient `∝ rᵀ`: **vanishes** (`r<1`) or **explodes** (`r>1`). We *measured* a real RNN's gradient vanish to ~`10⁻¹³`.
- The **LSTM cell state** is an *additive* highway with forget gate `f≈1`, so `∂cₜ/∂cₜ₋₁≈1` — the gradient **survives** (constant error carousel). Same measurement, flat line.
- Gates **store / hold / overwrite**; the highway costs ~**4×** the parameters (GRU ~**3×**).
- **Vanishing → better architecture; exploding → gradient clipping.**